# BlueRaven Telemetry Visualization Notebook

This notebook turns raw CSV telemetry into clear plots and checks.

What you get:
- quick data sanity checks
- timeline and packet sync view
- gyroscope and accelerometer trends
- quaternion quality check
- power system trends (voltage/current)
- simple anomaly flags

In [54]:
# Install plotting/data packages if they are missing.
# Safe to rerun: it only installs missing packages.
import importlib
import subprocess
import sys

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "plotly": "plotly",
    "nbformat": "nbformat>=4.2.0",
    "IPython": "ipython",
}

missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are already installed.")

All required packages are already installed.


In [55]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Force a VS Code/Jupyter-safe renderer to avoid nbformat mime issues.
pio.renderers.default = "notebook_connected"

pd.set_option("display.max_columns", 200)

# Try common locations so the notebook works no matter where it is launched from.
filename = "Level 3 High Rate BlueRaven (4).csv"
candidates = [
    Path(filename),
    Path("./backend/TestData") / filename,
    Path("../TestData") / filename,
]

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(f"Could not find {filename}. Checked: {[str(p) for p in candidates]}")

print(f"Loading: {data_path.resolve()}")
df = pd.read_csv(data_path)

numeric_cols = [
    "Flight_Time_(s)", "Sync",
    "Gyro_X", "Gyro_Y", "Gyro_Z",
    "Accel_X", "Accel_Y", "Accel_Z",
    "Quat_1", "Quat_2", "Quat_3", "Quat_4",
    "Aux_Volts", "Current",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Build an absolute timestamp from the split date/time columns.
date_str = (
    df["Year"].astype("Int64").astype(str).str.zfill(4)
    + "-" + df["Month"].astype("Int64").astype(str).str.zfill(2)
    + "-" + df["Day"].astype("Int64").astype(str).str.zfill(2)
)
df["timestamp"] = pd.to_datetime(date_str + " " + df["Time"].astype(str), errors="coerce")

# Derived metrics make the signal easier to reason about.
df["gyro_mag"] = np.sqrt(df["Gyro_X"]**2 + df["Gyro_Y"]**2 + df["Gyro_Z"]**2)
df["accel_mag"] = np.sqrt(df["Accel_X"]**2 + df["Accel_Y"]**2 + df["Accel_Z"]**2)
df["quat_norm"] = np.sqrt(df["Quat_1"]**2 + df["Quat_2"]**2 + df["Quat_3"]**2 + df["Quat_4"]**2)

# Downsample large datasets to keep interactive plots responsive.
max_points = 8000
if len(df) > max_points:
    step = int(np.ceil(len(df) / max_points))
    viz_df = df.iloc[::step].copy()
else:
    step = 1
    viz_df = df.copy()

print(f"Rows loaded: {len(df):,} | Rows plotted: {len(viz_df):,} | Downsample step: {step}")

Loading: /Users/lehangajanayake/Projects/ARD/backend/TestData/Level 3 High Rate BlueRaven (4).csv
Rows loaded: 89,350 | Rows plotted: 7,446 | Downsample step: 12


In [56]:
# Quick health snapshot of the dataset.
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
}).sort_values(["missing", "dtype"], ascending=[False, True])

display(df.head())
display(summary)

describe_cols = ["Flight_Time_(s)", "gyro_mag", "accel_mag", "quat_norm", "Aux_Volts", "Current"]
display(df[describe_cols].describe().T)

,Year,Month,Day,Time,Flight_Time_(s),Sync,Gyro_X,Gyro_Y,Gyro_Z,Accel_X,Accel_Y,Accel_Z,Quat_1,Quat_2,Quat_3,Quat_4,Aux_Volts,Current,timestamp,gyro_mag,accel_mag,quat_norm
0,2025,9,1,13:08:00.149,-2.030,85,0.3,0.1,0.0,0.99,0.02,0.0,1.0,0.00000,0.0,0.0,0.064,0.101,2025-09-01 13:08:00.149,0.316228,0.990202,1.0
1,2025,9,1,13:08:00.151,-2.028,87,0.1,0.0,0.0,1.00,0.02,0.0,1.0,0.00000,0.0,0.0,0.064,0.096,2025-09-01 13:08:00.151,0.100000,1.000200,1.0
2,2025,9,1,13:08:00.153,-2.026,89,0.4,0.0,-0.1,0.99,0.01,0.0,1.0,0.00000,0.0,0.0,0.064,0.096,2025-09-01 13:08:00.153,0.412311,0.990051,1.0
3,2025,9,1,13:08:00.155,-2.024,91,0.4,0.1,0.0,0.99,0.02,0.0,1.0,0.00003,0.0,0.0,0.064,0.096,2025-09-01 13:08:00.155,0.412311,0.990202,1.0
4,2025,9,1,13:08:00.157,-2.022,93,0.2,0.0,0.0,0.99,0.03,0.0,1.0,0.00003,0.0,0.0,0.065,0.093,2025-09-01 13:08:00.157,0.200000,0.990454,1.0


,dtype,missing,missing_pct
timestamp,datetime64[ns],0,0.0
Flight_Time_(s),float64,0,0.0
Gyro_X,float64,0,0.0
Gyro_Y,float64,0,0.0
Gyro_Z,float64,0,0.0
Accel_X,float64,0,0.0
Accel_Y,float64,0,0.0
Accel_Z,float64,0,0.0
Quat_1,float64,0,0.0
Quat_2,float64,0,0.0


,count,mean,std,min,25%,50%,75%,max
Flight_Time_(s),89350.0,87.319000,51.586535,-2.030000,42.644500,87.319000,131.993500,176.668000
gyro_mag,89350.0,255.666810,169.522957,0.000000,123.569221,233.146874,360.375856,1167.737042
accel_mag,89350.0,1.751240,1.980976,0.000000,1.049131,1.564960,1.973778,405.519928
quat_norm,89350.0,0.999974,0.000011,0.999935,0.999966,0.999974,0.999982,1.000004
Aux_Volts,89350.0,0.064434,0.000624,0.058000,0.064000,0.064000,0.065000,0.071000
Current,89350.0,0.099809,0.006152,0.061000,0.098000,0.099000,0.103000,1.502000


In [57]:
# Timeline: flight clock and packet sync trend.
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=viz_df["timestamp"],
        y=viz_df["Flight_Time_(s)"],
        mode="lines",
        name="Flight_Time_(s)",
        line=dict(width=2),
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=viz_df["timestamp"],
        y=viz_df["Sync"],
        mode="lines",
        name="Sync",
        line=dict(width=1, dash="dot"),
        opacity=0.7,
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Timeline Overview",
    template="plotly_white",
    hovermode="x unified",
    height=450,
)
fig.update_xaxes(title_text="Timestamp")
fig.update_yaxes(title_text="Flight Time (s)", secondary_y=False)
fig.update_yaxes(title_text="Sync", secondary_y=True)
fig.show(renderer="browser")

In [58]:
# IMU trends: Gyroscope and Accelerometer (axes + magnitude).
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Gyroscope (X, Y, Z, magnitude)", "Accelerometer (X, Y, Z, magnitude)"),
)

for col in ["Gyro_X", "Gyro_Y", "Gyro_Z", "gyro_mag"]:
    fig.add_trace(
        go.Scatter(x=viz_df["timestamp"], y=viz_df[col], mode="lines", name=col),
        row=1,
        col=1,
    )

for col in ["Accel_X", "Accel_Y", "Accel_Z", "accel_mag"]:
    fig.add_trace(
        go.Scatter(x=viz_df["timestamp"], y=viz_df[col], mode="lines", name=col),
        row=2,
        col=1,
    )

fig.update_layout(
    title="IMU Signal Overview",
    template="plotly_white",
    hovermode="x unified",
    height=800,
)
fig.update_xaxes(title_text="Timestamp", row=2, col=1)
fig.update_yaxes(title_text="Gyro", row=1, col=1)
fig.update_yaxes(title_text="Accel", row=2, col=1)
fig.show(renderer="browser")

In [59]:
# Quaternion components and norm quality check.
fig = make_subplots(specs=[[{"secondary_y": True}]])

for col in ["Quat_1", "Quat_2", "Quat_3", "Quat_4"]:
    fig.add_trace(
        go.Scatter(x=viz_df["timestamp"], y=viz_df[col], mode="lines", name=col),
        secondary_y=False,
    )

fig.add_trace(
    go.Scatter(
        x=viz_df["timestamp"],
        y=viz_df["quat_norm"],
        mode="lines",
        name="quat_norm",
        line=dict(width=2, dash="dash"),
    ),
    secondary_y=True,
)

fig.add_hline(y=1.0, line_dash="dot", line_color="black", opacity=0.6)
fig.update_layout(
    title="Quaternion Stability (norm should stay near 1)",
    template="plotly_white",
    hovermode="x unified",
    height=500,
)
fig.update_xaxes(title_text="Timestamp")
fig.update_yaxes(title_text="Quaternion components", secondary_y=False)
fig.update_yaxes(title_text="Quaternion norm", secondary_y=True)
fig.show(renderer="browser")

In [60]:
# Power system trends.
power_fig = make_subplots(specs=[[{"secondary_y": True}]])

power_fig.add_trace(
    go.Scatter(
        x=viz_df["timestamp"],
        y=viz_df["Aux_Volts"],
        mode="lines",
        name="Aux_Volts",
        line=dict(width=2),
    ),
    secondary_y=False,
)
power_fig.add_trace(
    go.Scatter(
        x=viz_df["timestamp"],
        y=viz_df["Current"],
        mode="lines",
        name="Current",
        line=dict(width=2, dash="dot"),
    ),
    secondary_y=True,
)

power_fig.update_layout(
    title="Power Overview",
    template="plotly_white",
    hovermode="x unified",
    height=450,
)
power_fig.update_xaxes(title_text="Timestamp")
power_fig.update_yaxes(title_text="Aux_Volts", secondary_y=False)
power_fig.update_yaxes(title_text="Current", secondary_y=True)
power_fig.show(renderer="browser")

In [61]:
# Simple anomaly flags for quick triage (adjust thresholds as needed).
flags = pd.DataFrame(index=df.index)
flags["high_gyro"] = df["gyro_mag"] > 5.0
flags["high_accel"] = df["accel_mag"] > 2.5
flags["quat_norm_drift"] = (df["quat_norm"] - 1.0).abs() > 0.02
flags["low_voltage"] = df["Aux_Volts"] < 0.05

flag_counts = flags.sum().sort_values(ascending=False).rename("count")
flag_pct = (flags.mean() * 100).round(3).rename("percent")
flag_summary = pd.concat([flag_counts, flag_pct], axis=1)

print("Anomaly Summary")
display(flag_summary)

# Show a small sample of rows where at least one flag is active.
active_rows = flags.any(axis=1)
interesting = df.loc[active_rows, [
    "timestamp", "Flight_Time_(s)", "gyro_mag", "accel_mag", "quat_norm", "Aux_Volts", "Current"
]].head(20)

display(interesting)

Anomaly Summary


,count,percent
high_gyro,86913,97.273
high_accel,10995,12.306
quat_norm_drift,0,0.000
low_voltage,0,0.000


,timestamp,Flight_Time_(s),gyro_mag,accel_mag,quat_norm,Aux_Volts,Current
942,2025-09-01 13:08:02.033,-0.146,1.232883,3.202202,0.999970,0.064,0.093
943,2025-09-01 13:08:02.035,-0.144,7.452516,8.497782,0.999970,0.064,0.095
944,2025-09-01 13:08:02.037,-0.142,5.124451,8.412199,0.999970,0.064,0.101
945,2025-09-01 13:08:02.039,-0.140,3.271085,8.477741,0.999970,0.064,0.096
946,2025-09-01 13:08:02.041,-0.138,4.318565,8.720189,0.999970,0.064,0.096
947,2025-09-01 13:08:02.043,-0.136,7.198611,9.533730,0.999970,0.064,0.098
948,2025-09-01 13:08:02.045,-0.134,11.365738,9.263590,0.999970,0.064,0.098
949,2025-09-01 13:08:02.047,-0.132,3.638681,9.683006,0.999970,0.064,0.096
950,2025-09-01 13:08:02.049,-0.130,8.675252,10.054695,0.999970,0.065,0.101
951,2025-09-01 13:08:02.051,-0.128,13.828955,9.783282,0.999970,0.065,0.101


## How to use

1. Run cells top to bottom.
2. If package install happens, rerun the import/loading cell once.
3. If plots feel heavy, lower `max_points` in the data-loading cell.
4. Tune anomaly thresholds in the last cell to match your vehicle behavior.

## Flight Story (What Happened)

This section infers key moments from IMU + quaternion data:
- launch window
- powered segment (estimated)
- apogee (from reconstructed vertical position)
- descent segment

Note: The 3D path is a dead-reckoned reconstruction from IMU integration, so drift is expected. It is best for understanding shape and phases, not exact GPS-level position.

In [65]:
# Build an explainable flight-phase model from IMU + attitude.
analysis_df = df.copy().sort_values("timestamp").reset_index(drop=True)

# Time base in seconds from first valid timestamp.
analysis_df["t"] = (analysis_df["timestamp"] - analysis_df["timestamp"].iloc[0]).dt.total_seconds()
analysis_df["dt"] = analysis_df["t"].diff().fillna(0.0).clip(lower=0.0)

# Normalize quaternions.
q = analysis_df[["Quat_1", "Quat_2", "Quat_3", "Quat_4"]].to_numpy(dtype=float)
q_norm = np.linalg.norm(q, axis=1)
q_norm[q_norm == 0] = 1.0
q = q / q_norm[:, None]

w = q[:, 0]
x = q[:, 1]
y = q[:, 2]
z = q[:, 3]

# Rotation matrix (body -> world) from quaternion.
r11 = 1 - 2 * (y * y + z * z)
r12 = 2 * (x * y - z * w)
r13 = 2 * (x * z + y * w)
r21 = 2 * (x * y + z * w)
r22 = 1 - 2 * (x * x + z * z)
r23 = 2 * (y * z - x * w)
r31 = 2 * (x * z - y * w)
r32 = 2 * (y * z + x * w)
r33 = 1 - 2 * (x * x + y * y)

acc_body = analysis_df[["Accel_X", "Accel_Y", "Accel_Z"]].to_numpy(dtype=float)

acc_world = np.column_stack([
    r11 * acc_body[:, 0] + r12 * acc_body[:, 1] + r13 * acc_body[:, 2],
    r21 * acc_body[:, 0] + r22 * acc_body[:, 1] + r23 * acc_body[:, 2],
    r31 * acc_body[:, 0] + r32 * acc_body[:, 1] + r33 * acc_body[:, 2],
])

# Remove gravity and convert g -> m/s^2.
lin_acc_world_g = acc_world - np.array([0.0, 0.0, 1.0])
lin_acc_world = lin_acc_world_g * 9.80665
analysis_df["lin_acc_mag_g"] = np.linalg.norm(lin_acc_world_g, axis=1)

# Event detection anchors.
launch_idx = int((analysis_df["Flight_Time_(s)"] >= 0).idxmax()) if (analysis_df["Flight_Time_(s)"] >= 0).any() else 0
prelaunch_mask = analysis_df["Flight_Time_(s)"] < 0

# Robust rest-state statistics.
accel_mag = analysis_df["accel_mag"]
prelaunch_acc = accel_mag[prelaunch_mask]
baseline = float(prelaunch_acc.median()) if len(prelaunch_acc) else float(accel_mag.median())
reference = prelaunch_acc if len(prelaunch_acc) else accel_mag
mad = float(np.median(np.abs(reference - np.median(reference))))
robust_noise = 1.4826 * mad
powered_threshold = baseline + max(0.10, 4.0 * robust_noise)

analysis_df["powered_like"] = accel_mag.rolling(7, min_periods=1).mean() > powered_threshold
powered_candidates = analysis_df.index[(analysis_df.index >= launch_idx) & analysis_df["powered_like"]]
powered_start_idx = int(powered_candidates[0]) if len(powered_candidates) else launch_idx

settled = accel_mag.rolling(12, min_periods=1).mean() < (baseline + max(0.05, 2.0 * robust_noise))
after_start = analysis_df.index > powered_start_idx
burn_end_candidates = analysis_df.index[after_start & settled]
burn_end_idx = int(burn_end_candidates[0]) if len(burn_end_candidates) else min(powered_start_idx + 80, len(analysis_df) - 1)

gyro_pre = analysis_df.loc[prelaunch_mask, "gyro_mag"]
gyro_baseline = float(gyro_pre.median()) if len(gyro_pre) else float(analysis_df["gyro_mag"].median())
gyro_ref = gyro_pre if len(gyro_pre) else analysis_df["gyro_mag"]
gyro_mad = float(np.median(np.abs(gyro_ref - np.median(gyro_ref))))
gyro_noise = 1.4826 * gyro_mad

accel_near_rest = (analysis_df["accel_mag"] - baseline).abs() < max(0.05, 2.5 * robust_noise)
gyro_near_rest = analysis_df["gyro_mag"].rolling(15, min_periods=1).mean() < (gyro_baseline + max(0.2, 4.0 * gyro_noise))
rest_like = accel_near_rest & gyro_near_rest

# Bias correction using pre-launch stationary window.
prelaunch_idx = np.where(prelaunch_mask.to_numpy())[0]
if len(prelaunch_idx) > 20:
    acc_bias_world = np.median(lin_acc_world[prelaunch_idx], axis=0)
else:
    seed_n = min(200, len(lin_acc_world))
    acc_bias_world = np.median(lin_acc_world[:seed_n], axis=0)

lin_acc_world_corr = lin_acc_world - acc_bias_world
analysis_df["lin_acc_z_ms2"] = lin_acc_world_corr[:, 2]

# Integrate acceleration -> velocity -> position.
dt = analysis_df["dt"].to_numpy()
vel_uncorrected = np.zeros_like(lin_acc_world_corr)
vel = np.zeros_like(lin_acc_world_corr)
pos = np.zeros_like(lin_acc_world_corr)

leak = 0.9995
zupt_gain = 0.25

for i in range(1, len(analysis_df)):
    # Uncorrected baseline integration (for comparison only).
    vel_uncorrected[i] = vel_uncorrected[i - 1] + 0.5 * (lin_acc_world_corr[i] + lin_acc_world_corr[i - 1]) * dt[i]

    # Drift-limited integration.
    v_next = vel[i - 1] + 0.5 * (lin_acc_world_corr[i] + lin_acc_world_corr[i - 1]) * dt[i]
    v_next *= leak

    if i >= launch_idx and bool(rest_like.iloc[i]):
        v_next *= (1.0 - zupt_gain)

    vel[i] = v_next
    pos[i] = pos[i - 1] + 0.5 * (vel[i] + vel[i - 1]) * dt[i]

analysis_df[["vx", "vy", "vz"]] = vel
analysis_df[["x", "y", "z"]] = pos
analysis_df["vz_uncorrected"] = vel_uncorrected[:, 2]
analysis_df["vz_corrected"] = vel[:, 2]

# Apogee and descent from corrected trajectory.
z_smooth = analysis_df["z"].rolling(25, center=True, min_periods=1).mean()
post_launch_idx = analysis_df.index[analysis_df.index >= launch_idx]
apogee_idx = int(z_smooth.loc[post_launch_idx].idxmax())
if apogee_idx <= launch_idx:
    apogee_idx = burn_end_idx

vz_smooth = analysis_df["vz_corrected"].rolling(25, center=True, min_periods=1).mean()
descent_candidates = analysis_df.index[(analysis_df.index > apogee_idx) & (vz_smooth < 0)]
descent_start_idx = int(descent_candidates[0]) if len(descent_candidates) else apogee_idx

# Landing detection: sustained quiet state after descent.
vz_near_rest = vz_smooth.abs() < 1.5
median_dt = float(np.nanmedian(analysis_df["dt"].replace(0, np.nan)))
if not np.isfinite(median_dt) or median_dt <= 0:
    median_dt = 0.01
steady_window = max(20, int(round(1.0 / median_dt)))

landing_like = rest_like & vz_near_rest
landing_sustained = landing_like.rolling(steady_window, min_periods=steady_window).sum() >= steady_window
landing_candidates = analysis_df.index[(analysis_df.index > descent_start_idx) & landing_sustained]
landing_idx = int(landing_candidates[0]) if len(landing_candidates) else len(analysis_df) - 1

events = pd.DataFrame([
    {"event": "Launch (Flight_Time >= 0)", "index": launch_idx},
    {"event": "Powered Start (estimated)", "index": powered_start_idx},
    {"event": "Powered End / Coast Start (estimated)", "index": burn_end_idx},
    {"event": "Apogee (reconstructed z peak)", "index": apogee_idx},
    {"event": "Descent Start (estimated)", "index": descent_start_idx},
    {"event": "Landing (estimated)", "index": landing_idx},
])

events["timestamp"] = events["index"].map(analysis_df["timestamp"])
events["flight_time_s"] = events["index"].map(analysis_df["Flight_Time_(s)"])
display(events[["event", "timestamp", "flight_time_s"]])

print("Interpretation notes:")
print(f"- Baseline accel magnitude: {baseline:.3f} g")
print(f"- Robust accel noise:       {robust_noise:.3f} g")
print(f"- Powered threshold used:   {powered_threshold:.3f} g")
print(f"- Baseline gyro magnitude:  {gyro_baseline:.3f}")
print(f"- Landing steady window:    {steady_window} samples (~{steady_window * median_dt:.2f} s)")
print(f"- Estimated accel bias (m/s^2): [{acc_bias_world[0]:.3f}, {acc_bias_world[1]:.3f}, {acc_bias_world[2]:.3f}]")
print("- Velocity now uses drift-limited integration with ZUPT-like damping during rest.")

landing_time = analysis_df.loc[landing_idx, "timestamp"]
landing_flight_time = float(analysis_df.loc[landing_idx, "Flight_Time_(s)"])
print(f"\nEstimated landing time: {landing_time} (Flight_Time ~= {landing_flight_time:.3f} s)")

# Convenience cut: keep only through landing + short buffer.
post_buffer_s = 2.0
cut_flight_time = landing_flight_time + post_buffer_s
trimmed_df = df[df["Flight_Time_(s)"] <= cut_flight_time].copy()
print(f"Suggested trim cutoff: Flight_Time_(s) <= {cut_flight_time:.3f}")
print(f"Trimmed rows: {len(trimmed_df):,} / {len(df):,}")

,event,timestamp,flight_time_s
0,Launch (Flight_Time >= 0),2025-09-01 13:08:02.179,0.000
1,Powered Start (estimated),2025-09-01 13:08:02.179,0.000
2,Powered End / Coast Start (estimated),2025-09-01 13:08:12.223,10.044
3,Apogee (reconstructed z peak),2025-09-01 13:08:12.223,10.044
4,Descent Start (estimated),2025-09-01 13:08:12.225,10.046
5,Landing (estimated),2025-09-01 13:10:58.157,175.978


Interpretation notes:
- Baseline accel magnitude: 0.990 g
- Robust accel noise:       0.014 g
- Powered threshold used:   1.090 g
- Baseline gyro magnitude:  0.245
- Landing steady window:    500 samples (~1.00 s)
- Estimated accel bias (m/s^2): [9.709, 0.196, -9.807]
- Velocity now uses drift-limited integration with ZUPT-like damping during rest.

Estimated landing time: 2025-09-01 13:10:58.157000 (Flight_Time ~= 175.978 s)
Suggested trim cutoff: Flight_Time_(s) <= 177.978
Trimmed rows: 89,350 / 89,350


In [66]:
# Explainable timeline with phase shading + event markers.
story_df = analysis_df.copy()

# Downsample for plotting performance.
story_step = max(1, int(np.ceil(len(story_df) / 10000)))
story_viz = story_df.iloc[::story_step].copy()

story_fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        "Flight clock + reconstructed altitude (z)",
        "Linear acceleration magnitude (g)",
        "Vertical velocity (m/s): corrected vs uncorrected",
    ),
)

story_fig.add_trace(
    go.Scatter(x=story_viz["timestamp"], y=story_viz["Flight_Time_(s)"], mode="lines", name="Flight_Time_(s)"),
    row=1, col=1,
)
story_fig.add_trace(
    go.Scatter(x=story_viz["timestamp"], y=story_viz["z"], mode="lines", name="Reconstructed z (m)", line=dict(dash="dot")),
    row=1, col=1,
)
story_fig.add_trace(
    go.Scatter(x=story_viz["timestamp"], y=story_viz["lin_acc_mag_g"], mode="lines", name="Linear accel |a| (g)"),
    row=2, col=1,
)
story_fig.add_trace(
    go.Scatter(x=story_viz["timestamp"], y=story_viz["vz_corrected"], mode="lines", name="vz_corrected (m/s)", line=dict(width=2.5)),
    row=3, col=1,
)
story_fig.add_trace(
    go.Scatter(x=story_viz["timestamp"], y=story_viz["vz_uncorrected"], mode="lines", name="vz_uncorrected (m/s)", line=dict(width=1, dash="dot"), opacity=0.45),
    row=3, col=1,
)

# Phase shading.
powered_start_t = analysis_df.loc[powered_start_idx, "timestamp"]
burn_end_t = analysis_df.loc[burn_end_idx, "timestamp"]
apogee_t = analysis_df.loc[apogee_idx, "timestamp"]
landing_t = analysis_df.loc[landing_idx, "timestamp"]

story_fig.add_vrect(x0=powered_start_t, x1=burn_end_t, fillcolor="orange", opacity=0.18, line_width=0, annotation_text="Powered", annotation_position="top left")
story_fig.add_vrect(x0=burn_end_t, x1=apogee_t, fillcolor="royalblue", opacity=0.10, line_width=0, annotation_text="Coast", annotation_position="top left")
story_fig.add_vrect(x0=apogee_t, x1=landing_t, fillcolor="seagreen", opacity=0.08, line_width=0, annotation_text="Descent", annotation_position="top left")
story_fig.add_vrect(x0=landing_t, x1=analysis_df["timestamp"].iloc[-1], fillcolor="gray", opacity=0.08, line_width=0, annotation_text="Post-flight / likely noise", annotation_position="top left")

for _, ev in events.iterrows():
    story_fig.add_vline(x=ev["timestamp"], line_dash="dot", line_color="black", opacity=0.5)

story_fig.update_layout(
    title="Flight Story: inferred phases and key events",
    template="plotly_white",
    hovermode="x unified",
    height=900,
)
story_fig.update_xaxes(title_text="Timestamp", row=3, col=1)
story_fig.update_yaxes(title_text="Time / z", row=1, col=1)
story_fig.update_yaxes(title_text="|a| (g)", row=2, col=1)
story_fig.update_yaxes(title_text="vz (m/s)", row=3, col=1)
story_fig.show(renderer="browser")

# Compact text summary.
print("\nFlight story summary:")
print(f"- Launch near Flight_Time = {analysis_df.loc[launch_idx, 'Flight_Time_(s)']:.3f} s")
print(f"- Powered segment estimate: {analysis_df.loc[powered_start_idx, 'Flight_Time_(s)']:.3f} s to {analysis_df.loc[burn_end_idx, 'Flight_Time_(s)']:.3f} s")
print(f"- Apogee estimate at Flight_Time = {analysis_df.loc[apogee_idx, 'Flight_Time_(s)']:.3f} s")
print(f"- Descent starts around Flight_Time = {analysis_df.loc[descent_start_idx, 'Flight_Time_(s)']:.3f} s")
print(f"- Landing estimate at Flight_Time = {analysis_df.loc[landing_idx, 'Flight_Time_(s)']:.3f} s")


Flight story summary:
- Launch near Flight_Time = 0.000 s
- Powered segment estimate: 0.000 s to 10.044 s
- Apogee estimate at Flight_Time = 10.044 s
- Descent starts around Flight_Time = 10.046 s
- Landing estimate at Flight_Time = 175.978 s


In [64]:
# 3D reconstructed flight path (dead reckoning from IMU + quaternion).
traj_step = max(1, int(np.ceil(len(analysis_df) / 12000)))
traj = analysis_df.iloc[::traj_step].copy()

traj_fig = go.Figure()
traj_fig.add_trace(
    go.Scatter3d(
        x=traj["x"],
        y=traj["y"],
        z=traj["z"],
        mode="lines",
        line=dict(width=5, color=traj["Flight_Time_(s)"], colorscale="Turbo", colorbar=dict(title="Flight_Time (s)")),
        name="Trajectory",
    )
)

# Mark only the key points requested.
def event_point(name):
    idx = int(events.loc[events["event"] == name, "index"].iloc[0])
    row = analysis_df.loc[idx]
    return row["x"], row["y"], row["z"]

for name, marker, size in [
    ("Launch (Flight_Time >= 0)", "circle", 7),
    ("Apogee (reconstructed z peak)", "diamond", 9),
]:
    ex, ey, ez = event_point(name)
    traj_fig.add_trace(
        go.Scatter3d(
            x=[ex], y=[ey], z=[ez],
            mode="markers+text",
            marker=dict(size=size, symbol=marker, color="black"),
            text=[name.split(" (")[0]],
            textposition="top center",
            name=name.split(" (")[0],
        )
    )

traj_fig.update_layout(
    title="3D Reconstructed Flight Path (Launch and Apogee)",
    template="plotly_white",
    scene=dict(
        xaxis_title="X (m, relative)",
        yaxis_title="Y (m, relative)",
        zaxis_title="Z (m, relative)",
        aspectmode="data",
    ),
    height=750,
)
traj_fig.show(renderer="browser")

print("3D chart note: this is a relative path estimate from IMU integration, useful for shape/phases, not precise absolute position.")

3D chart note: this is a relative path estimate from IMU integration, useful for shape/phases, not precise absolute position.
